## Simulação do portal com LinUCB

Este notebook simula o loop de recomendação em produção com o LinUCB. Um usuário chega, o modelo lista as ofertas elegíveis ranqueadas para o perfil dele, exibimos a oferta do topo, recebemos um evento de clique, atualizamos o modelo e mostramos a próxima oferta.

O clique é gerado por um **ambiente de simulação interno** (um modelo de preferência oculto do usuário), e não pelo `offer_catalog.json`. O catálogo aqui é a versão de produção: ele não carrega parâmetros de simulação, só os campos de serving. Para usar cliques reais, basta trocar o ambiente de simulação pelo log do front-end.

## Como funciona

O fluxo tem duas peças bem separadas:

`LinUCB` é a **política** na qual recebe o contexto do usuário e devolve um ranking de ofertas e aprende com cada clique.

`SimuladorUsuario` é o **ambiente** no qual guarda a preferência verdadeira do usuário e decide se ele clica. O LinUCB nunca enxerga essa preferência, ele só observa cliques e tenta recuperá-la. 

obs: em produção, essa peça some e o clique vem do usuário real.

## Setup: catálogo, clientes e elegibilidade

In [1]:
import json, ast
import numpy as np
import pandas as pd

In [2]:
np.random.seed(42)

DATA_DIR = "../data/golden_set"
catalog = json.load(open(f"{DATA_DIR}/offer_catalog.json", encoding="utf-8"))
df = pd.read_csv(f"{DATA_DIR}/golden_clients.csv")
offers = catalog["offers"]
df["_renda_pct"] = df["renda_estimada_anual_brl"].rank(pct=True) * 100

In [3]:
def parse_segs(client):
    """Separa os segmentos sintéticos do cliente em um set de strings"""
    try: 
        return set(ast.literal_eval(client["segmentos_sinteticos"]))
    except Exception: 
        return set()

def is_eligible(client, filters):
    """Verifica se o cliente atende aos filtros de elegibilidade da oferta"""
    for key, val in filters.items():
        if key.endswith("_atual"):
            if client.get(key[:-6], 0) != val: return False
        elif key.endswith("_percentil_min"):
            if client["_renda_pct"] < val: return False
        elif key.endswith("_min"):
            if client.get(key[:-4], 0) < val: return False
        elif key.endswith("_max"):
            if client.get(key[:-4], 1e18) > val: return False
        else:
            if client.get(key) != val: return False
    return True

In [4]:
clients = df.to_dict("records")
client_segs = [parse_segs(c) for c in clients]
E = np.array([[is_eligible(c, o["eligible_segment"]["santander_filters"]) for o in offers] for c in clients])

print(f"{len(offers)} ofertas no catálogo, {len(df):,} clientes")
print(f"Elegibilidade média: {E.sum(1).mean():.1f} ofertas por cliente")

10 ofertas no catálogo, 2,595 clientes
Elegibilidade média: 5.4 ofertas por cliente


## Perfil do usuário (vetor de contexto)

O LinUCB enxerga o usuário como um vetor `x`: as `context_features` numéricas padronizadas, mais um intercepto, mais o one hot dos segmentos sintéticos.

In [5]:
seg_keys = ['SEG-JOVEM','SEG-SENIOR','SEG-ALTA-RENDA','SEG-VIP','SEG-CREDITO-ATIVO',
            'SEG-INVESTIDOR-EXPERIENTE','SEG-PERFIL-FAMILIAR','SEG-POUPADOR',
            'SEG-CONTRIBUINTE-IR','SEG-INVESTIDOR-INICIANTE','SEG-SEM-CARTAO','SEG-PROPRIETARIO']

In [6]:
ctx_cols = offers[0]["context_features"]
_mu = df[ctx_cols].astype(float).mean()
_sd = df[ctx_cols].astype(float).std()

def context_vec(i):
    """Retorna o vetor de contexto do cliente i, concatenando os atributos numéricos e os segmentos sintéticos"""
    num = ((df[ctx_cols].iloc[i].astype(float) - _mu) / _sd).values
    seg = np.array([1.0 if k in client_segs[i] else 0.0 for k in seg_keys])
    return np.concatenate([[1.0], num, seg])

D = len(context_vec(0))
print(f"Dimensão do contexto: {D}  (1 bias, {len(ctx_cols)} numéricas, {len(seg_keys)} segmentos)")

Dimensão do contexto: 22  (1 bias, 9 numéricas, 12 segmentos)


## Ambiente de simulação (gera os cliques)

O ambiente guarda, por braço, um vetor de peso sobre o contexto. A probabilidade verdadeira de clique é uma logística desse peso aplicado ao perfil do usuário. Como o peso é fixo, o comportamento é reprodutível, e o LinUCB tem um sinal real e contextual para aprender.

Para produção: precisamos substituir `clicar` por uma leitura do evento real (ex:`/feedback`) e remover `prob_verdadeira`.

In [7]:
class SimuladorUsuario:
    """Ambiente de simulação. Representa a preferência verdadeira do usuário."""
    def __init__(self, n_arms, d, seed=123):
        rng = np.random.default_rng(seed)
        self.W    = rng.normal(0.0, 0.35, size=(n_arms, d))   # peso latente por braço
        self.bias = rng.normal(-2.0, 0.30, size=n_arms)        # base baixa (cliques são raros)

    def prob_verdadeira(self, j, x):
        z = self.W[j] @ x + self.bias[j]
        return 1.0 / (1.0 + np.exp(-z))

    def clicar(self, j, x):
        """Evento de clique observado: 1 (clicou) ou 0."""
        return int(np.random.random() < self.prob_verdadeira(j, x))

ambiente = SimuladorUsuario(len(offers), D, seed=123)

In [8]:
# checagem rápida da taxa de clique gerada
taxas = [ambiente.prob_verdadeira(j, context_vec(i))
         for i in range(len(df)) for j in range(len(offers)) if E[i, j]]
print(f"Clique simulado: médio={np.mean(taxas):.3f}, "
      f"mediana={np.median(taxas):.3f}, faixa=[{np.percentile(taxas,10):.3f}, {np.percentile(taxas,90):.3f}]")

Clique simulado: médio=0.220, mediana=0.153, faixa=[0.049, 0.484]


## LinUCB: serving e atualização

`ranquear` devolve as ofertas elegíveis ordenadas pelo score UCB, que soma a predição de clique e um bônus de exploração. `atualizar` incorpora o clique observado via Sherman Morrison, sem inverter matriz a cada passo.

In [9]:
class LinUCB:
    def __init__(self, n_arms, d, exploration, scale=0.2, lam=1.0):
        self.alpha = [e * scale for e in exploration]          # ucb_params escalado
        self.A_inv = [np.eye(d) / lam for _ in range(n_arms)]  # cold start via ridge
        self.b     = [np.zeros(d)     for _ in range(n_arms)]

    def _score(self, j, x):
        theta = self.A_inv[j] @ self.b[j]
        pred  = theta @ x
        bonus = self.alpha[j] * np.sqrt(max(x @ self.A_inv[j] @ x, 1e-9))
        return pred + bonus, pred, bonus

    def ranquear(self, x, eligible, excluir=()):
        linhas = []
        for j in range(len(eligible)):
            if not eligible[j] or j in excluir: continue
            s, pred, bonus = self._score(j, x)
            linhas.append((j, s, pred, bonus))
        return sorted(linhas, key=lambda r: -r[1])

    def atualizar(self, j, reward, x):
        Ax = self.A_inv[j] @ x
        self.A_inv[j] -= np.outer(Ax, Ax) / (1.0 + x @ Ax)
        self.b[j]     += reward * x

exploration = [o["ucb_params"]["exploration_factor"] for o in offers]
modelo = LinUCB(len(offers), D, exploration)
print("LinUCB inicializado (cold-start).")

LinUCB inicializado (cold-start).


## Histórico acumulado (warm-up)

Em produção o modelo já viu tráfego. Simulamos esse histórico rodando o loop sobre uma amostra de usuários, para que as recomendações já saiam personalizadas. Comece sem esta célula para ver o modelo aprendendo do zero.

In [10]:
valid = np.where(E.sum(1) > 0)[0]

for _ in range(3000):
    i = valid[np.random.randint(len(valid))]
    x = context_vec(i)
    topo = modelo.ranquear(x, E[i])[0][0]
    modelo.atualizar(topo, ambiente.clicar(topo, x), x)
    
print("Warm-up concluído (3.000 interações). Modelo pronto para servir.")

Warm-up concluído (3.000 interações). Modelo pronto para servir.


## Funções de serving

`listar_ofertas` é o que o `/decide` devolveria ao front-end: a vitrine ranqueada para o perfil. `simular_sessao` roda o ciclo exibir, clicar, recalcular, próxima oferta.

In [11]:
def perfil(i):
    c = clients[i]
    return (f"Cliente {c['cod_cliente']}, {c['idade']} anos, "
            f"renda R$ {c['renda_estimada_anual_brl']:,.0f}, "
            f"segmentos: {', '.join(sorted(client_segs[i]))}")

def listar_ofertas(i, excluir=(), top=None):
    rk = modelo.ranquear(context_vec(i), E[i], excluir=excluir)
    linhas = [{
        "rank": r+1, "arm_id": offers[j]["arm_id"], "produto": offers[j]["product_name"],
        "categoria": offers[j]["category"], "score_ucb": round(s, 3),
        "pred_clique": round(pred, 3), "bonus_expl": round(bonus, 3),
    } for r, (j, s, pred, bonus) in enumerate(rk)]
    out = pd.DataFrame(linhas)
    return out.head(top) if top else out

def simular_sessao(i, n_interacoes=5, seed=None):
    if seed is not None: np.random.seed(seed)
    print(perfil(i)); print(f"{int(E[i].sum())} ofertas elegíveis\n")
    x = context_vec(i); exibidas = set()
    for passo in range(n_interacoes):
        rk = modelo.ranquear(x, E[i], excluir=exibidas)
        if not rk:
            print("Sem mais ofertas elegíveis nesta sessão."); break
        print(f"[Interação {passo+1}] ranking LinUCB (top 3):")
        for (j, s, pred, bonus) in rk[:3]:
            print(f"    {offers[j]['arm_id']:<12} {offers[j]['product_name']:<32} "
                  f"score={s:.3f}  (pred={pred:+.3f}, expl={bonus:.3f})")
        j = rk[0][0]
        clique = ambiente.clicar(j, x)
        print(f"  exibida: {offers[j]['product_name']}")
        print(f"  evento de clique: {'CLICOU' if clique else 'não clicou'} (reward={clique})")
        modelo.atualizar(j, clique, x); exibidas.add(j)
        print(f"  LinUCB recalculado.\n")

## Simulação: usuário A

Vitrine inicial, ou seja, o que o front-end receberia, e depois a sessão interativa.

In [12]:
i_A = next(i for i in valid if E[i].sum() >= 6)   # usuário com vitrine rica
print(perfil(i_A), "\n")
listar_ofertas(i_A)

Cliente 100870, 45 anos, renda R$ 47,821, segmentos: SEG-CONTRIBUINTE-IR, SEG-CREDITO-ATIVO, SEG-INVESTIDOR-INICIANTE, SEG-PERFIL-FAMILIAR, SEG-POUPADOR 



,rank,arm_id,produto,categoria,score_ucb,pred_clique,bonus_expl
0,1,OFF-INV-002,CDB Médio Prazo,investimento,0.668,0.631,0.038
1,2,OFF-SEG-003,Seguro Viagem,seguro,0.494,0.417,0.077
2,3,OFF-SEG-001,Seguro de Vida Familiar,seguro,0.425,0.062,0.364
3,4,OFF-INV-004,Fundo Multimercado,investimento,0.334,0.175,0.159
4,5,OFF-CR-001,Crédito Pessoal Pré-Aprovado,credito,0.249,0.016,0.233
5,6,OFF-INV-001,CDB Primeiros Passos,investimento,-0.104,-0.248,0.144


In [13]:
simular_sessao(i_A, n_interacoes=5, seed=10)

Cliente 100870, 45 anos, renda R$ 47,821, segmentos: SEG-CONTRIBUINTE-IR, SEG-CREDITO-ATIVO, SEG-INVESTIDOR-INICIANTE, SEG-PERFIL-FAMILIAR, SEG-POUPADOR
6 ofertas elegíveis

[Interação 1] ranking LinUCB (top 3):
    OFF-INV-002  CDB Médio Prazo                  score=0.668  (pred=+0.631, expl=0.038)
    OFF-SEG-003  Seguro Viagem                    score=0.494  (pred=+0.417, expl=0.077)
    OFF-SEG-001  Seguro de Vida Familiar          score=0.425  (pred=+0.062, expl=0.364)
  exibida: CDB Médio Prazo
  evento de clique: não clicou (reward=0)
  LinUCB recalculado.

[Interação 2] ranking LinUCB (top 3):
    OFF-SEG-003  Seguro Viagem                    score=0.494  (pred=+0.417, expl=0.077)
    OFF-SEG-001  Seguro de Vida Familiar          score=0.425  (pred=+0.062, expl=0.364)
    OFF-INV-004  Fundo Multimercado               score=0.334  (pred=+0.175, expl=0.159)
  exibida: Seguro Viagem
  evento de clique: CLICOU (reward=1)
  LinUCB recalculado.

[Interação 3] ranking LinUCB (top 3):


## Simulação: usuário B (perfil diferente)

Mesmo modelo, perfil distinto, logo vitrine e trajetória diferentes. É a personalização contextual em ação.

In [14]:
i_B = next(i for i in valid
           if 'SEG-JOVEM' in client_segs[i] and 'SEG-SEM-CARTAO' in client_segs[i]
           and E[i].sum() >= 5)
print(perfil(i_B), "\n")
listar_ofertas(i_B, top=5)

Cliente 1186912, 23 anos, renda R$ 146,222, segmentos: SEG-CREDITO-ATIVO, SEG-INVESTIDOR-INICIANTE, SEG-JOVEM, SEG-POUPADOR, SEG-SEM-CARTAO 



,rank,arm_id,produto,categoria,score_ucb,pred_clique,bonus_expl
0,1,OFF-INV-002,CDB Médio Prazo,investimento,0.611,0.588,0.023
1,2,OFF-INV-003,Previdência Privada PGBL,investimento,0.508,0.445,0.063
2,3,OFF-CR-002,Cartão de Crédito Mais,credito,0.306,0.237,0.068
3,4,OFF-INV-004,Fundo Multimercado,investimento,0.226,0.031,0.195
4,5,OFF-SEG-003,Seguro Viagem,seguro,0.157,0.067,0.089


In [15]:
simular_sessao(i_B, n_interacoes=5, seed=20)

Cliente 1186912, 23 anos, renda R$ 146,222, segmentos: SEG-CREDITO-ATIVO, SEG-INVESTIDOR-INICIANTE, SEG-JOVEM, SEG-POUPADOR, SEG-SEM-CARTAO
7 ofertas elegíveis

[Interação 1] ranking LinUCB (top 3):
    OFF-INV-002  CDB Médio Prazo                  score=0.611  (pred=+0.588, expl=0.023)
    OFF-INV-003  Previdência Privada PGBL         score=0.508  (pred=+0.445, expl=0.063)
    OFF-CR-002   Cartão de Crédito Mais           score=0.306  (pred=+0.237, expl=0.068)
  exibida: CDB Médio Prazo
  evento de clique: não clicou (reward=0)
  LinUCB recalculado.

[Interação 2] ranking LinUCB (top 3):
    OFF-INV-003  Previdência Privada PGBL         score=0.508  (pred=+0.445, expl=0.063)
    OFF-CR-002   Cartão de Crédito Mais           score=0.306  (pred=+0.237, expl=0.068)
    OFF-INV-004  Fundo Multimercado               score=0.226  (pred=+0.031, expl=0.195)
  exibida: Previdência Privada PGBL
  evento de clique: não clicou (reward=0)
  LinUCB recalculado.

[Interação 3] ranking LinUCB (top 3)